## Problem Statement - Implementation of Clonal selection algorithm using Python.
**Sidhi Pawar BE B - 64**

In [4]:
import numpy as np

In [5]:
class CLONALG:
    def __init__(self, pop_size=20, bounds=[-5.0, 5.0], generations=50, clone_multiplier=5, mutation_rate=0.2):
        self.pop_size = pop_size
        self.bounds = bounds
        self.generations = generations
        self.clone_multiplier = clone_multiplier
        self.mutation_rate = mutation_rate
        self.num_variables = 2 # (x, y) coordinates
        
    # --- CORE PRINCIPLES ---
    
    def affinity_function(self, antibody):
        """
        Evaluates the fitness of a candidate solution.
        Aim: Maximize the function f(x, y) = -(x^2 + y^2) + 100.
        The maximum affinity is 100, which occurs precisely at x=0, y=0.
        """
        x, y = antibody
        return -(x**2 + y**2) + 100

    def calculate_population_affinity(self, population):
        """Helper function to evaluate an entire population."""
        return np.array([self.affinity_function(ind) for ind in population])

    # --- CLONALG PHASES ---

    def optimize(self):
        # 1. Initialization: Generate a random population of antibodies
        population = np.random.uniform(
            self.bounds[0], self.bounds[1], 
            (self.pop_size, self.num_variables)
        )
        
        print("Starting CLONALG Optimization...\n")

        for gen in range(self.generations):
            # 2. Evaluation
            affinities = self.calculate_population_affinity(population)

            # 3. Selection (Sort descending to find high-affinity antibodies)
            sorted_indices = np.argsort(affinities)[::-1]
            population = population[sorted_indices]
            affinities = affinities[sorted_indices]

            # 4. Cloning: Create clones proportional to affinity rank
            clones = []
            for i in range(self.pop_size):
                # Higher affinity (lower index i) gets more clones
                num_clones = int((self.clone_multiplier * self.pop_size) / (i + 1))
                for _ in range(num_clones):
                    clones.append(population[i].copy())
            clones = np.array(clones)

            # 5. Hypermutation: Introduce random mutations to clones
            for i in range(len(clones)):
                if np.random.rand() < self.mutation_rate:
                    # Apply a random shift to the coordinates
                    mutation_step = np.random.normal(0, 0.5, self.num_variables)
                    clones[i] += mutation_step
                    # Keep the mutated clones within our defined search bounds
                    clones[i] = np.clip(clones[i], self.bounds[0], self.bounds[1])

            # 6. Evaluation (of mutated clones)
            clone_affinities = self.calculate_population_affinity(clones)

            # 7. Selection and Replacement
            # Combine original population and mutated clones
            combined_population = np.vstack((population, clones))
            combined_affinities = np.concatenate((affinities, clone_affinities))

            # Select the absolute best to form the next generation (elitism)
            best_indices = np.argsort(combined_affinities)[::-1][:self.pop_size]
            population = combined_population[best_indices]
            best_affinity = combined_affinities[best_indices[0]]

            # Print progress every 10 generations
            if (gen + 1) % 10 == 0:
                print(f"Generation {gen + 1:2d} | Best Affinity: {best_affinity:.4f} | Best Antibody: {population[0]}")

        # 8. Termination
        print("\nOptimization Complete.")
        best_antibody = population[0]
        final_affinity = self.calculate_population_affinity([best_antibody])[0]
        return best_antibody, final_affinity



In [6]:
# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    # Initialize and run the algorithm
    immune_system = CLONALG(pop_size=20, generations=50)
    best_solution, best_fitness = immune_system.optimize()
    
    print("\n--- Final Results ---")
    print(f"Ideal Target     : x=0.0, y=0.0 (Affinity: 100.0)")
    print(f"Algorithm Found  : x={best_solution[0]:.4f}, y={best_solution[1]:.4f} (Affinity: {best_fitness:.4f})")

Starting CLONALG Optimization...

Generation 10 | Best Affinity: 99.9983 | Best Antibody: [ 0.01826853 -0.03673829]
Generation 20 | Best Affinity: 99.9997 | Best Antibody: [-0.01372948  0.01220961]
Generation 30 | Best Affinity: 99.9997 | Best Antibody: [0.01529791 0.00663687]
Generation 40 | Best Affinity: 99.9997 | Best Antibody: [0.01529791 0.00663687]
Generation 50 | Best Affinity: 99.9998 | Best Antibody: [0.0123321  0.00181199]

Optimization Complete.

--- Final Results ---
Ideal Target     : x=0.0, y=0.0 (Affinity: 100.0)
Algorithm Found  : x=0.0123, y=0.0018 (Affinity: 99.9998)
